## Setup

In [1]:
import pandas as pd
import re

## Load Datasets

In [2]:
df = pd.read_csv(r"../clean_data/ulta_clean_blush_v1.csv", encoding="utf-8-sig")
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 1514 entries, 0 to 1513
Data columns (total 14 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   shade_url          1514 non-null   str    
 1   brand              1514 non-null   str    
 2   product_name       1514 non-null   str    
 3   shade              1514 non-null   str    
 4   swatch_img_url     1514 non-null   str    
 5   swatch_alt         1514 non-null   str    
 6   description        1148 non-null   str    
 7   price              1514 non-null   float64
 8   currency           1514 non-null   str    
 9   standard_value     1514 non-null   float64
 10  standard_unit      1514 non-null   str    
 11  product_id         1507 non-null   float64
 12  shade_id           1514 non-null   int64  
 13  product_image_url  1514 non-null   str    
dtypes: float64(3), int64(1), str(10)
memory usage: 165.7 KB


## Create New Columns

Since the dataset does not have ID numbers for product and shade and image URLs of the product, we create these data by extracting from the `shade_url`. For example: "https://www.ulta.com/p/skin-perfecting-powder-blushing-act-matte-blush-pimprod2012166?sku=2560675" 
- product_id: **2012166**
- sku_id or shade_id: **2560675**
- product_image_url:  "https://media.ulta.com/i/ulta/2560675?w=1080&h=1080&fmt=auto"

In [ ]:
# 1. Extract Product ID (the digits following 'pimprod')
# Regex logic: find 'pimprod' and capture the digits (\d+) that follow it
# df['product_id'] = df['shade_url'].str.extract(r'(?:pimprod|mkt|xlsImpprod)(\d+)') # (?:rod|mkt)(\d+)
df['product_id'] = df['shade_url'].str.extract(r'(?:rod|mkt|VP)(\d+)')

# 2. Extract Shade ID / SKU (the digits following 'sku=')
# Regex logic: find 'sku=' and capture the digits (\d+) that follow it
df['shade_id'] = df['shade_url'].str.extract(r'sku=(\d+)')

# Optional: Convert them to strings or integers
# (Sometimes it's better to keep IDs as strings to prevent dropping leading zeros)
df['product_id'] = df['product_id'].astype(str)
df['shade_id'] = df['shade_id'].astype(str)

In [4]:
# See which URLs aren't playing nice
df.loc[df['shade_id'].isna(), 'shade_url'].unique()

<StringArray>
[]
Length: 0, dtype: str

In [ ]:
# Define the base and suffix
base_url = "https://media.ulta.com/i/ulta/" # the main address
suffix = "?w=1080&h=1080&fmt=auto" # the image size

# Create the column using an f-string style join (Vectorized)
# We only create URLs where shade_id is NOT 'unknown'
df['product_image_url'] = df['shade_id'].apply(
    lambda x: f"{base_url}{x}{suffix}" if x != 'unknown' else None
)

# To see your 'empty' count (where shade_id was unknown)
empty_count = df['product_image_url'].isna().sum()
print(f"Number of products without image URLs: {empty_count}")

Number of products without image URLs: 0


In [9]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 1514 entries, 0 to 1513
Data columns (total 14 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   shade_url          1514 non-null   str    
 1   brand              1514 non-null   str    
 2   product_name       1514 non-null   str    
 3   shade              1514 non-null   str    
 4   swatch_img_url     1514 non-null   str    
 5   swatch_alt         1514 non-null   str    
 6   description        1148 non-null   str    
 7   price              1514 non-null   float64
 8   currency           1514 non-null   str    
 9   standard_value     1514 non-null   float64
 10  standard_unit      1514 non-null   str    
 11  product_id         1507 non-null   str    
 12  shade_id           1514 non-null   str    
 13  product_image_url  1514 non-null   str    
dtypes: float64(2), str(12)
memory usage: 165.7 KB


## Export

In [ ]:
# df.to_csv(r"../clean_data/ulta_clean_blush_v1.csv", encoding='utf-8-sig', index=False)

## Remove Column `currency`

In [19]:
df = df.drop(['currency'], axis=1)

In [20]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 1514 entries, 0 to 1513
Data columns (total 13 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   shade_url          1514 non-null   str    
 1   brand              1514 non-null   str    
 2   product_name       1514 non-null   str    
 3   shade              1514 non-null   str    
 4   swatch_img_url     1514 non-null   str    
 5   swatch_alt         1514 non-null   str    
 6   description        1148 non-null   str    
 7   price              1514 non-null   float64
 8   standard_value     1514 non-null   float64
 9   standard_unit      1514 non-null   str    
 10  product_id         1507 non-null   float64
 11  shade_id           1514 non-null   int64  
 12  product_image_url  1514 non-null   str    
dtypes: float64(3), int64(1), str(9)
memory usage: 153.9 KB


## Handle Missing Values in `description`

In [32]:
print(f"Total Length of description: {len(df.description)}")
print(f"Total Values of description: {len(df.description)-len(df.loc[df.description.isnull()])}")
print(f"Total Missing Values of description: {len(df.loc[df.description.isnull()])}")

Total Length of description: 1514
Total Values of description: 1148
Total Missing Values of description: 366


In [33]:
## Display the products with missing values
df.loc[df.description.isnull(), ~df.columns.isin(['price','standard_value', 'standard_unit'])]


,shade_url,brand,product_name,shade,swatch_img_url,swatch_alt,description,product_id,shade_id,product_image_url
27,https://www.ulta.com/p/pressed-powder-blush-pi...,Revlon,Pressed Powder Blush,Tickled Pink,https://media.ultainc.com/i/ulta/2545657_sw?im...,Tickled Pink Pressed Powder Blush,NaN,2007329,2545657,https://media.ulta.com/i/ulta/2545657?w=1080&h...
28,https://www.ulta.com/p/pressed-powder-blush-pi...,Revlon,Pressed Powder Blush,Apricute,https://media.ultainc.com/i/ulta/2546601_sw?im...,Apricute Pressed Powder Blush,NaN,2007329,2546601,https://media.ulta.com/i/ulta/2546601?w=1080&h...
29,https://www.ulta.com/p/pressed-powder-blush-pi...,Revlon,Pressed Powder Blush,Mauvelous,https://media.ultainc.com/i/ulta/2267589_sw?im...,Mauvelous Pressed Powder Blush,NaN,2007329,2267589,https://media.ulta.com/i/ulta/2267589?w=1080&h...
38,https://www.ulta.com/p/true-match-super-blenda...,L'Oréal,True Match Super Blendable Blush,Baby Blossom C1-2,https://media.ultainc.com/i/ulta/2114143_sw?im...,Baby Blossom C1-2 True Match Super Blendable B...,NaN,10730,2114143,https://media.ulta.com/i/ulta/2114143?w=1080&h...
39,https://www.ulta.com/p/true-match-super-blenda...,L'Oréal,True Match Super Blendable Blush,Rosy Outlook C5-6,https://media.ultainc.com/i/ulta/2114145_sw?im...,Rosy Outlook C5-6 True Match Super Blendable B...,NaN,10730,2114145,https://media.ulta.com/i/ulta/2114145?w=1080&h...
...,...,...,...,...,...,...,...,...,...,...
1487,https://www.ulta.com/p/value-size-liquid-lip-b...,Benefit Cosmetics,Value Size Liquid Lip Blush & Cheek Tint,Rose-tinted,https://media.ultainc.com/i/ulta/2620380_sw?im...,Rose-tinted Value Size Liquid Lip Blush & Chee...,NaN,2043819,2620380,https://media.ulta.com/i/ulta/2620380?w=1080&h...
1510,https://www.ulta.com/p/clean-fresh-all-over-de...,CoverGirl,Clean Fresh All Over Dewy Tint,Fuchsia Passion,https://media.ultainc.com/i/ulta/2597351_sw?im...,Fuchsia Passion Clean Fresh All Over Dewy Tint,NaN,2033619,2597351,https://media.ulta.com/i/ulta/2597351?w=1080&h...
1511,https://www.ulta.com/p/clean-fresh-all-over-de...,CoverGirl,Clean Fresh All Over Dewy Tint,Mauvy Kiss,https://media.ultainc.com/i/ulta/2597354_sw?im...,Mauvy Kiss Clean Fresh All Over Dewy Tint,NaN,2033619,2597354,https://media.ulta.com/i/ulta/2597354?w=1080&h...
1512,https://www.ulta.com/p/clean-fresh-all-over-de...,CoverGirl,Clean Fresh All Over Dewy Tint,Dreamy Pink,https://media.ultainc.com/i/ulta/2597355_sw?im...,Dreamy Pink Clean Fresh All Over Dewy Tint,NaN,2033619,2597355,https://media.ulta.com/i/ulta/2597355?w=1080&h...


***Conclusion***: 

After checking the URL of `description` with missing rows, these products just simply doesn't have color descriptions for their shades, like other 1148 shades.